## Análise inicial - Projeto do Ciclo Básico InsperData - Grupo 2
#### Integrantes:  Laura, Matias e Vinícius


Importação das bibliotecas necessárias:


In [ ]:
# Verificação/instalação de dependências para modelagem estatística
# Rode esta célula antes dos imports principais.

import sys
import subprocess
import importlib.util

def ensure_package(package_name, import_name=None):
    import_name = import_name or package_name
    if importlib.util.find_spec(import_name) is None:
        print(f'Instalando {package_name}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package_name])
    else:
        print(f'{package_name} já está instalado.')

ensure_package('statsmodels', 'statsmodels')


In [ ]:
from nba_api.stats.endpoints import leaguegamelog
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm


In [ ]:
# Pegando os dados da última temporada
gamelog = leaguegamelog.LeagueGameLog(season="2024-25", season_type_all_star="Regular Season")
dataframe = gamelog.get_data_frames()[0] #lista com 1 df com os filtros selecionados, pegar esse único termo

dataframe.head()


In [ ]:
print(dataframe.columns)


In [ ]:
df = dataframe[['GAME_ID',
        'TEAM_ABBREVIATION', 
        'TEAM_NAME', 
        'GAME_DATE', 
        'MATCHUP', # se for @ é fora de casa pro X @ x; se for vs é em casa pra X vs x;
        'WL', # TARGET
        'PTS', 
        'FG_PCT', # porcentagem de aproveitamento
        'FG3_PCT', # porcentagem de aproveitamento de 3 
        'FT_PCT', # porcentagem de aproveitamento de lance livre
        'PLUS_MINUS', # saldo dos pontos do jogo
        'OREB', # rebotes ofensivos (segunda chance de ataque)
        'AST', # assistências
        'TOV']].copy() # erros ou perdas de posse

df.dtypes


In [ ]:
# atualizando tipos de variáveis
df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE']) # transformando em data

# retorna 1 se o jogo é em casa para o primeiro time do MATCHUP; 0 se é fora de casa
df['HOME_AWAY'] = df['MATCHUP'].apply(lambda x: 1 if 'vs' in x else 0)
df['HOME_AWAY'] = df['HOME_AWAY'].astype('category')

# transformando vitória/derrota em variável binária: vitória = 1, derrota = 0
df['WL'] = df['WL'].map({'W': 1, 'L': 0}).astype(int)

# extraindo a sigla do adversário a partir do MATCHUP
# exemplos: "BOS vs. NYK" -> NYK; "BOS @ NYK" -> NYK
df['OPPONENT_ABBREVIATION'] = df['MATCHUP'].str.extract(r'(?:vs\.?|@)\s+([A-Z]{2,3})', expand=False)

df.dtypes


In [ ]:
# organizando o df para que os jogos de times diferentes não fiquem misturados e a ordem das datas esteja coerente
df = df.sort_values(['TEAM_ABBREVIATION', 'GAME_DATE', 'GAME_ID']).copy()


In [ ]:
# ------------------------------------------------------------
# Cálculo do descanso do time, descanso do adversário e força prévia
# ------------------------------------------------------------
# Como o LeagueGameLog possui duas linhas por partida, uma para cada time,
# primeiro calculamos o descanso de cada time individualmente e depois
# fazemos merges para trazer o descanso e a força prévia do oponente na mesma partida.

# Garantindo ordenação cronológica por time antes de calcular descanso e métricas prévias
df = df.sort_values(['TEAM_ABBREVIATION', 'GAME_DATE', 'GAME_ID']).copy()

# ------------------------------------------------------------
# Força prévia do time antes de cada jogo
# ------------------------------------------------------------
# Importante: calculamos isso ANTES de remover jogos sem descanso calculável,
# para que a campanha prévia use todos os jogos anteriores da temporada.

df['TEAM_GAMES_BEFORE'] = df.groupby('TEAM_ABBREVIATION').cumcount()

df['TEAM_WINS_BEFORE'] = (
    df.groupby('TEAM_ABBREVIATION')['WL']
    .transform(lambda s: s.cumsum().shift(1))
)

df['TEAM_PLUS_MINUS_SUM_BEFORE'] = (
    df.groupby('TEAM_ABBREVIATION')['PLUS_MINUS']
    .transform(lambda s: s.cumsum().shift(1))
)

df['TEAM_WIN_PCT_BEFORE'] = df['TEAM_WINS_BEFORE'] / df['TEAM_GAMES_BEFORE']
df['TEAM_AVG_PLUS_MINUS_BEFORE'] = df['TEAM_PLUS_MINUS_SUM_BEFORE'] / df['TEAM_GAMES_BEFORE']

# Valores neutros para primeiros jogos, caso apareçam em alguma análise posterior.
df['TEAM_WIN_PCT_BEFORE'] = df['TEAM_WIN_PCT_BEFORE'].replace([np.inf, -np.inf], np.nan).fillna(0.5)
df['TEAM_AVG_PLUS_MINUS_BEFORE'] = df['TEAM_AVG_PLUS_MINUS_BEFORE'].replace([np.inf, -np.inf], np.nan).fillna(0)

# ------------------------------------------------------------
# Descanso do próprio time
# ------------------------------------------------------------
# coluna do último jogo para calcular os dias de descanso, agrupados por time
df['PREV_GAME'] = df.groupby('TEAM_ABBREVIATION')['GAME_DATE'].shift(1)
df['REST_DAYS_NUM'] = (df['GAME_DATE'] - df['PREV_GAME']).dt.days - 1

# Por segurança, valores negativos são tratados como 0.
# Isso pode ocorrer em situações raras de ordenação/metadata.
df.loc[df['REST_DAYS_NUM'] < 0, 'REST_DAYS_NUM'] = 0

def classificar_descanso(x):
    if pd.isna(x):
        return np.nan
    elif x == 0:
        return '0'
    elif x == 1:
        return '1'
    else:
        return '2+'

# descanso do próprio time em categorias: 0, 1 ou 2+
# Essa categorização é mantida para a variável detalhada REST_MATCHUP e para os heatmaps.
df['REST_DAYS'] = df['REST_DAYS_NUM'].apply(classificar_descanso)

# ------------------------------------------------------------
# Descanso e força prévia do adversário na mesma partida
# ------------------------------------------------------------
opponent_info = df[[
    'GAME_ID',
    'TEAM_ABBREVIATION',
    'REST_DAYS',
    'REST_DAYS_NUM',
    'TEAM_WIN_PCT_BEFORE',
    'TEAM_AVG_PLUS_MINUS_BEFORE'
]].copy()

opponent_info = opponent_info.rename(columns={
    'TEAM_ABBREVIATION': 'OPPONENT_ABBREVIATION',
    'REST_DAYS': 'OPP_REST_DAYS',
    'REST_DAYS_NUM': 'OPP_REST_DAYS_NUM',
    'TEAM_WIN_PCT_BEFORE': 'OPP_WIN_PCT_BEFORE',
    'TEAM_AVG_PLUS_MINUS_BEFORE': 'OPP_AVG_PLUS_MINUS_BEFORE'
})

# trazendo para cada linha o descanso e a força prévia do adversário naquela mesma partida
df = df.merge(
    opponent_info,
    on=['GAME_ID', 'OPPONENT_ABBREVIATION'],
    how='left'
)

# removendo jogos em que o time ou o adversário ainda não tinha jogo anterior na temporada
# Esses jogos não têm descanso prévio interpretável.
df = df.dropna(subset=['REST_DAYS', 'OPP_REST_DAYS', 'REST_DAYS_NUM', 'OPP_REST_DAYS_NUM']).copy()

# Diferenças de força pré-jogo: positivo favorece o time analisado
df['WIN_PCT_DIFF_BEFORE'] = df['TEAM_WIN_PCT_BEFORE'] - df['OPP_WIN_PCT_BEFORE']
df['AVG_PLUS_MINUS_DIFF_BEFORE'] = df['TEAM_AVG_PLUS_MINUS_BEFORE'] - df['OPP_AVG_PLUS_MINUS_BEFORE']

# mantendo a ordem lógica das categorias
rest_categories = ['0', '1', '2+']
df['REST_DAYS'] = pd.Categorical(df['REST_DAYS'], categories=rest_categories, ordered=True)
df['OPP_REST_DAYS'] = pd.Categorical(df['OPP_REST_DAYS'], categories=rest_categories, ordered=True)

# variável detalhada: descanso do time vs descanso do oponente
# exemplo: "0 vs 2+" = time em back-to-back contra adversário com 2+ dias de descanso
# Atenção: essa variável é categórica e compacta valores >= 2 em "2+".
df['REST_MATCHUP'] = df['REST_DAYS'].astype(str) + ' vs ' + df['OPP_REST_DAYS'].astype(str)

rest_matchup_order = [
    '0 vs 0', '0 vs 1', '0 vs 2+',
    '1 vs 0', '1 vs 1', '1 vs 2+',
    '2+ vs 0', '2+ vs 1', '2+ vs 2+'
]
df['REST_MATCHUP'] = pd.Categorical(df['REST_MATCHUP'], categories=rest_matchup_order, ordered=True)

# ------------------------------------------------------------
# Diferença REAL de descanso para os gráficos de barras
# ------------------------------------------------------------
# Aqui NÃO usamos a categoria "2+" para calcular a diferença.
# Usamos os valores numéricos reais:
# - 1 vs 2 = 1 dia de diferença
# - 1 vs 3 = 2+ dias de diferença
# - 2 vs 5 = 2+ dias de diferença
# - 3 vs 3 = descanso igual

df['REST_ADVANTAGE'] = df['REST_DAYS_NUM'] - df['OPP_REST_DAYS_NUM']
df['REST_DIFF_ABS'] = df['REST_ADVANTAGE'].abs()

def classificar_diferenca_real(diff):
    if pd.isna(diff):
        return np.nan
    elif diff == 0:
        return 'Descanso igual'
    elif diff == 1:
        return '1 Dia de descanso'
    else:
        return '2+ Dias de descanso'

rest_comparison_order = [
    'Descanso igual',
    '1 Dia de descanso',
    '2+ Dias de descanso'
]

df['REST_COMPARISON_GROUP'] = df['REST_DIFF_ABS'].apply(classificar_diferenca_real)
df['REST_COMPARISON_GROUP'] = pd.Categorical(
    df['REST_COMPARISON_GROUP'],
    categories=rest_comparison_order,
    ordered=True
)

# Variáveis numéricas categorizadas apenas para possíveis análises complementares/heatmaps.
# Não são usadas para definir os grupos dos gráficos de barras.
rest_to_num_capped = {'0': 0, '1': 1, '2+': 2}
df['REST_DAYS_NUM_CAT'] = df['REST_DAYS'].astype(str).map(rest_to_num_capped)
df['OPP_REST_DAYS_NUM_CAT'] = df['OPP_REST_DAYS'].astype(str).map(rest_to_num_capped)

df.head()


In [ ]:
# coluna com 1 para os jogos em que o time analisado está em back-to-back
df['BACK_TO_BACK'] = (df['REST_DAYS'] == '0').astype(int)

# coluna opcional: back-to-back contra adversário mais descansado
df['B2B_VS_RESTED_OPP'] = ((df['REST_DAYS'] == '0') & (df['OPP_REST_DAYS'].isin(['1', '2+']))).astype(int)

df.head()


In [ ]:
# conferindo a base após cálculo de descanso, resultado binário e variáveis principais
df.head()


In [ ]:
# removendo jogos a partir do dia 20 de março de 2025, quando começam as eliminações matemáticas dos playoffs e aumenta o incentivo ao tanking


In [ ]:
df.to_csv('nba_descanso_modelagem_ajustada.csv', index=False)


### Análises iniciais


#### Base dos gráficos de barras: uma linha por jogo

Para evitar que o grupo **Descanso igual** fique artificialmente com saldo médio 0, os gráficos de desempenho passam a usar uma base com apenas uma linha por partida:

- quando há diferença de descanso, mantém-se a linha do **time com menor descanso**;
- quando o descanso é igual, mantém-se apenas a linha do **mandante**.

As análises detalhadas em heatmap e os modelos estatísticos continuam usando suas respectivas bases próprias.

In [ ]:
# ------------------------------------------------------------
# Base para gráficos de barras com UMA linha por jogo
# ------------------------------------------------------------
# O LeagueGameLog tem duas linhas por partida: uma para cada time.
# Se mantivermos as duas linhas no grupo "Descanso igual", o saldo médio
# zera por construção: +saldo do vencedor e -saldo do perdedor.
#
# Para os gráficos de desempenho:
# - jogos com diferença de descanso: mantemos o time com MENOR descanso;
# - jogos com descanso igual: mantemos apenas o MANDANTE.

df_barras_desempenho = df.copy()

# HOME_AWAY pode estar como categoria; convertemos para int para filtrar com segurança.
df_barras_desempenho['HOME_AWAY_INT'] = df_barras_desempenho['HOME_AWAY'].astype(int)

cond_menor_descanso = (
    df_barras_desempenho['REST_DAYS_NUM'] < df_barras_desempenho['OPP_REST_DAYS_NUM']
)

cond_descanso_igual_mandante = (
    (df_barras_desempenho['REST_DAYS_NUM'] == df_barras_desempenho['OPP_REST_DAYS_NUM']) &
    (df_barras_desempenho['HOME_AWAY_INT'] == 1)
)

df_barras_desempenho = df_barras_desempenho[
    cond_menor_descanso | cond_descanso_igual_mandante
].copy()

df_barras_desempenho['GRAFICO_PERSPECTIVA'] = np.where(
    df_barras_desempenho['REST_DIFF_ABS'] == 0,
    'Mandante em jogos com descanso igual',
    'Time com menor descanso'
)

# Checagem: cada GAME_ID deve aparecer apenas uma vez nessa base.
duplicados_grafico = df_barras_desempenho['GAME_ID'].duplicated().sum()

print(f'Jogos na base original após filtros de descanso: {df["GAME_ID"].nunique()}')
print(f'Jogos na base dos gráficos: {df_barras_desempenho["GAME_ID"].nunique()}')
print(f'Linhas duplicadas por GAME_ID na base dos gráficos: {duplicados_grafico}')

df_barras_desempenho[
    ['GAME_ID', 'TEAM_ABBREVIATION', 'OPPONENT_ABBREVIATION', 'HOME_AWAY_INT',
     'REST_DAYS_NUM', 'OPP_REST_DAYS_NUM', 'REST_COMPARISON_GROUP',
     'GRAFICO_PERSPECTIVA', 'WL', 'PLUS_MINUS']
].head()


In [ ]:
# Distribuição dos jogos por diferença real de descanso
# Usa a mesma base dos gráficos de desempenho, com uma linha por GAME_ID.
df_barras_jogos = df_barras_desempenho.copy()

plt.figure(figsize=(10,6))
sns.countplot(
    x='REST_COMPARISON_GROUP',
    data=df_barras_jogos,
    stat='percent',
    palette='muted',
    order=rest_comparison_order
)
plt.title('Distribuição dos jogos por diferença real de descanso', loc='center', fontsize=15)
plt.xlabel('Diferença de descanso entre os times', fontsize=13, loc='center')
plt.ylabel('Percentual dos jogos (%)', fontsize=13, loc='center')
plt.xticks(rotation=0, ha='center')
plt.grid(visible=True, alpha=0.7, ls='--', axis='y')
plt.tight_layout()
plt.savefig('grafico_descanso_diferenca_real_agrupada.jpg')
plt.show()


In [ ]:
# Taxa de vitória por diferença real de descanso
# Usa uma linha por jogo:
# - diferença de descanso: perspectiva do time com menor descanso;
# - descanso igual: perspectiva do mandante.

plt.figure(figsize=(10,6))
sns.barplot(
    x='REST_COMPARISON_GROUP',
    y='WL',
    data=df_barras_desempenho,
    palette='muted',
    errorbar=None,
    order=rest_comparison_order
)
plt.title('Taxa de vitória por diferença real de descanso', fontsize=15, loc='center')
plt.xlabel('Diferença de descanso entre os times', fontsize=13, loc='center')
plt.ylabel('Proporção de vitórias', fontsize=13, loc='center')
plt.ylim(0,1)
plt.xticks(rotation=0, ha='center')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('vitoria_por_diferenca_real_descanso_agrupada.jpg')
plt.show()


In [ ]:
# Saldo médio por diferença real de descanso
# Usa uma linha por jogo:
# - diferença de descanso: perspectiva do time com menor descanso;
# - descanso igual: perspectiva do mandante.
plt.figure(figsize=(10,6))
sns.barplot(
    x='REST_COMPARISON_GROUP',
    y='PLUS_MINUS',
    data=df_barras_desempenho,
    palette='muted',
    errorbar=None,
    order=rest_comparison_order
)
plt.axhline(0, color='black', linestyle='--', alpha=0.5)
plt.title('Saldo médio por diferença real de descanso', fontsize=15, loc='center')
plt.xlabel('Diferença de descanso entre os times', fontsize=13, loc='center')
plt.ylabel('Saldo médio do jogo', fontsize=13, loc='center')
plt.xticks(rotation=0, ha='center')
plt.grid(axis='y', alpha=0.7)
plt.tight_layout()
plt.savefig('saldo_medio_por_diferenca_real_descanso_agrupada.jpg')
plt.show()


#### Resumo numérico por diferença real de descanso


In [ ]:
# Resumo numérico usando os mesmos grupos dos gráficos de barras.
# Esta base já tem uma linha por jogo:
# - diferença de descanso: time com menor descanso;
# - descanso igual: mandante.
resumo_descanso_oponente = (
    df_barras_desempenho.groupby('REST_COMPARISON_GROUP', observed=False)
    .agg(
        jogos=('GAME_ID', 'count'),
        vitorias=('WL', 'sum'),
        taxa_vitoria=('WL', 'mean'),
        pontos_marcados=('PTS', 'mean'),
        saldo_medio=('PLUS_MINUS', 'mean'),
        aproveitamento_arremessos=('FG_PCT', 'mean'),
        aproveitamento_3pts=('FG3_PCT', 'mean'),
        rebotes_ofensivos=('OREB', 'mean'),
        assistencias=('AST', 'mean'),
        turnovers=('TOV', 'mean')
    )
    .reindex(rest_comparison_order)
    .reset_index()
)

resumo_descanso_oponente['taxa_vitoria'] = resumo_descanso_oponente['taxa_vitoria'] * 100
resumo_descanso_oponente


#### Matrizes: descanso do time x descanso do oponente


In [ ]:
# matriz de taxa de vitória
matriz_vitoria = (
    df.groupby(['REST_DAYS', 'OPP_REST_DAYS'], observed=False)['WL']
    .mean()
    .mul(100)
    .unstack()
    .reindex(index=rest_categories, columns=rest_categories)
)

plt.figure(figsize=(8,6))
sns.heatmap(matriz_vitoria, annot=True, fmt='.1f', cmap='Blues')
plt.title('Taxa de vitória (%) - descanso do time x descanso do oponente', fontsize=14)
plt.xlabel('Descanso do oponente')
plt.ylabel('Descanso do time')
plt.tight_layout()
plt.savefig('heatmap_taxa_vitoria_descanso_oponente.jpg')
plt.show()


In [ ]:
# matriz de saldo médio
matriz_saldo = (
    df.groupby(['REST_DAYS', 'OPP_REST_DAYS'], observed=False)['PLUS_MINUS']
    .mean()
    .unstack()
    .reindex(index=rest_categories, columns=rest_categories)
)

plt.figure(figsize=(8,6))
sns.heatmap(matriz_saldo, annot=True, fmt='.2f', cmap='RdBu_r', center=0)
plt.title('Saldo médio - descanso do time x descanso do oponente', fontsize=14)
plt.xlabel('Descanso do oponente')
plt.ylabel('Descanso do time')
plt.tight_layout()
plt.savefig('heatmap_saldo_descanso_oponente.jpg')
plt.show()


In [ ]:
# matriz de volume de jogos
matriz_jogos = (
    df.groupby(['REST_DAYS', 'OPP_REST_DAYS'], observed=False)['GAME_ID']
    .count()
    .unstack()
    .reindex(index=rest_categories, columns=rest_categories)
)

plt.figure(figsize=(8,6))
sns.heatmap(matriz_jogos, annot=True, fmt='.0f', cmap='Greens')
plt.title('Número de jogos - descanso do time x descanso do oponente', fontsize=14)
plt.xlabel('Descanso do oponente')
plt.ylabel('Descanso do time')
plt.tight_layout()
plt.savefig('heatmap_jogos_descanso_oponente.jpg')
plt.show()


In [ ]:
plt.figure(figsize=(8,6))
sns.boxplot(x=df.HOME_AWAY.map({0:'Fora de casa', 1: 'Casa'}), y='PLUS_MINUS', data=df, palette='muted')
plt.axhline(0, color='black', linestyle='--', alpha=0.5)
plt.title('Desempenho por local de jogo', fontsize=15, loc='center')
plt.xlabel('')
plt.ylabel('Saldo do jogo', fontsize=13, loc='center')
plt.grid(axis='y', alpha=0.7)
plt.savefig('desempenho_por_local_jogo.jpg')
plt.show()


### Modelagem estatística

Nesta etapa, o objetivo não é apenas prever vitória/derrota, mas estimar se a diferença de descanso entre os times está associada ao desempenho. Por isso, usamos modelos estatísticos com `statsmodels`, que facilitam a interpretação dos coeficientes, odds ratios e significância estatística.

A análise principal segue a mesma lógica dos gráficos: em jogos com descanso diferente, mantemos a perspectiva do time com menor descanso; em jogos com descanso igual, mantemos as duas linhas como referência neutra.

In [ ]:
# ------------------------------------------------------------
# Base para modelagem estatística
# ------------------------------------------------------------
# REST_ADVANTAGE = descanso do time - descanso do oponente.
# Para analisar o efeito de ter menos descanso, mantemos:
# - REST_ADVANTAGE < 0: time com menor descanso que o adversário;
# - REST_ADVANTAGE == 0: descanso igual.

model_df = df[df['REST_ADVANTAGE'] <= 0].copy()

# Rótulos mais claros para mando de quadra
model_df['HOME_AWAY_LABEL'] = model_df['HOME_AWAY'].astype(int).map({
    0: 'Fora',
    1: 'Casa'
})

# Garantindo ordem das categorias do descanso
model_df['REST_COMPARISON_GROUP'] = pd.Categorical(
    model_df['REST_COMPARISON_GROUP'],
    categories=rest_comparison_order,
    ordered=True
)

# Removendo eventuais linhas com dados ausentes nas variáveis dos modelos
model_df = model_df.dropna(subset=[
    'WL',
    'PLUS_MINUS',
    'REST_COMPARISON_GROUP',
    'HOME_AWAY_LABEL',
    'TEAM_WIN_PCT_BEFORE',
    'OPP_WIN_PCT_BEFORE',
    'TEAM_AVG_PLUS_MINUS_BEFORE',
    'OPP_AVG_PLUS_MINUS_BEFORE',
    'WIN_PCT_DIFF_BEFORE',
    'AVG_PLUS_MINUS_DIFF_BEFORE'
]).copy()

print(f'Observações time-jogo na modelagem: {len(model_df)}')
print(f'Jogos únicos representados: {model_df["GAME_ID"].nunique()}')
print()
print(model_df['REST_COMPARISON_GROUP'].value_counts().reindex(rest_comparison_order))


In [ ]:
# Funções auxiliares para ajustar modelos e resumir resultados

def remove_unused_categories(data):
    """Remove categorias não observadas para evitar colunas dummy vazias nos modelos."""
    data = data.copy()
    for col in data.select_dtypes(include=['category']).columns:
        data[col] = data[col].cat.remove_unused_categories()
    return data


def fit_logit_clustered(formula, data):
    """
    Regressão logística com erro-padrão agrupado por GAME_ID quando possível.

    Observação importante:
    - Em modelos com muitas categorias, como REST_MATCHUP, podem existir categorias
      sem observações após filtros da análise. Isso gera colunas dummy vazias e pode
      causar erro de matriz singular.
    - Por isso, removemos categorias não usadas antes do ajuste.
    - Se o Logit ainda falhar, usamos GLM Binomial como alternativa mais estável.
    """
    data_model = remove_unused_categories(data)

    try:
        return smf.logit(formula, data=data_model).fit(
            disp=False,
            cov_type='cluster',
            cov_kwds={'groups': data_model['GAME_ID']}
        )
    except Exception as e1:
        print(f'Aviso: não foi possível ajustar Logit com erro-padrão agrupado por GAME_ID. Erro: {e1}')

        try:
            return smf.logit(formula, data=data_model).fit(disp=False)
        except Exception as e2:
            print(f'Aviso: Logit padrão também falhou. Usando GLM Binomial. Erro: {e2}')

            try:
                return smf.glm(
                    formula=formula,
                    data=data_model,
                    family=sm.families.Binomial()
                ).fit(
                    cov_type='cluster',
                    cov_kwds={'groups': data_model['GAME_ID']}
                )
            except Exception as e3:
                print(f'Aviso: GLM Binomial agrupado também falhou. Usando GLM Binomial com HC3. Erro: {e3}')
                return smf.glm(
                    formula=formula,
                    data=data_model,
                    family=sm.families.Binomial()
                ).fit(cov_type='HC3')


def fit_ols_clustered(formula, data):
    # Regressão linear com erro-padrão agrupado por GAME_ID quando possível.
    data_model = remove_unused_categories(data)

    try:
        return smf.ols(formula, data=data_model).fit(
            cov_type='cluster',
            cov_kwds={'groups': data_model['GAME_ID']}
        )
    except Exception as e:
        print(f'Aviso: não foi possível usar erro-padrão agrupado por GAME_ID. Usando ajuste HC3. Erro: {e}')
        return smf.ols(formula, data=data_model).fit(cov_type='HC3')


def odds_ratio_table(model):
    # Tabela de odds ratios para modelos logísticos.
    conf = model.conf_int()
    tabela = pd.DataFrame({
        'coef_log_odds': model.params,
        'OR': np.exp(model.params),
        'IC95_inf_OR': np.exp(conf[0]),
        'IC95_sup_OR': np.exp(conf[1]),
        'p_valor': model.pvalues
    })
    return tabela


def coef_table(model):
    # Tabela resumida de coeficientes para modelos lineares.
    conf = model.conf_int()
    tabela = pd.DataFrame({
        'coef': model.params,
        'IC95_inf': conf[0],
        'IC95_sup': conf[1],
        'p_valor': model.pvalues
    })
    return tabela


def get_pseudo_r2(model):
    """Retorna pseudo-R² de McFadden quando disponível; caso contrário, retorna NaN."""
    return getattr(model, 'prsquared', np.nan)


#### Modelo 1 — efeito bruto da diferença de descanso

Este modelo testa a associação simples entre diferença de descanso e vitória, sem ajuste por outras variáveis.

In [ ]:
formula_logit_simples = """
WL ~ C(REST_COMPARISON_GROUP, Treatment(reference='Descanso igual'))
"""

modelo_logit_simples = fit_logit_clustered(formula_logit_simples, model_df)
odds_ratio_table(modelo_logit_simples)


#### Modelo 2 — ajuste por mando de quadra

Aqui adicionamos se o time estava jogando em casa ou fora, porque mando de quadra pode modificar desempenho independentemente do descanso.

In [ ]:
formula_logit_casa = """
WL ~ C(REST_COMPARISON_GROUP, Treatment(reference='Descanso igual'))
   + C(HOME_AWAY_LABEL, Treatment(reference='Fora'))
"""

modelo_logit_casa = fit_logit_clustered(formula_logit_casa, model_df)
odds_ratio_table(modelo_logit_casa)


#### Modelo 3 — ajuste por força prévia do time e do oponente

Este é o modelo principal sugerido. Ele controla por mando de quadra e por medidas de força dos times antes do jogo: aproveitamento prévio e saldo médio prévio do time e do oponente.

In [ ]:
formula_logit_ajustado = """
WL ~ C(REST_COMPARISON_GROUP, Treatment(reference='Descanso igual'))
   + C(HOME_AWAY_LABEL, Treatment(reference='Fora'))
   + TEAM_WIN_PCT_BEFORE
   + OPP_WIN_PCT_BEFORE
   + TEAM_AVG_PLUS_MINUS_BEFORE
   + OPP_AVG_PLUS_MINUS_BEFORE
"""

modelo_logit_ajustado = fit_logit_clustered(formula_logit_ajustado, model_df)
odds_ratio_table(modelo_logit_ajustado)


In [ ]:
# Probabilidades previstas de vitória pelo modelo ajustado
# Mantendo as variáveis de força no valor médio da amostra.

cenarios_descanso = pd.DataFrame({
    'REST_COMPARISON_GROUP': pd.Categorical(
        rest_comparison_order,
        categories=rest_comparison_order,
        ordered=True
    ),
    'HOME_AWAY_LABEL': 'Fora',
    'TEAM_WIN_PCT_BEFORE': model_df['TEAM_WIN_PCT_BEFORE'].mean(),
    'OPP_WIN_PCT_BEFORE': model_df['OPP_WIN_PCT_BEFORE'].mean(),
    'TEAM_AVG_PLUS_MINUS_BEFORE': model_df['TEAM_AVG_PLUS_MINUS_BEFORE'].mean(),
    'OPP_AVG_PLUS_MINUS_BEFORE': model_df['OPP_AVG_PLUS_MINUS_BEFORE'].mean()
})

cenarios_descanso['prob_vitoria_prevista'] = modelo_logit_ajustado.predict(cenarios_descanso)
cenarios_descanso[['REST_COMPARISON_GROUP', 'prob_vitoria_prevista']]


#### Modelo 4 — interação entre descanso e mando de quadra

Este modelo avalia se o efeito da diferença de descanso muda quando o time joga em casa ou fora.

In [ ]:
formula_logit_interacao = """
WL ~ C(REST_COMPARISON_GROUP, Treatment(reference='Descanso igual'))
   * C(HOME_AWAY_LABEL, Treatment(reference='Fora'))
   + TEAM_WIN_PCT_BEFORE
   + OPP_WIN_PCT_BEFORE
   + TEAM_AVG_PLUS_MINUS_BEFORE
   + OPP_AVG_PLUS_MINUS_BEFORE
"""

modelo_logit_interacao = fit_logit_clustered(formula_logit_interacao, model_df)
odds_ratio_table(modelo_logit_interacao)


#### Modelo 5 — saldo de pontos como desfecho contínuo

Além de vitória/derrota, é interessante testar o saldo de pontos (`PLUS_MINUS`), porque ele pode captar piora de desempenho mesmo quando o resultado final do jogo não muda.

In [ ]:
formula_ols_saldo = """
PLUS_MINUS ~ C(REST_COMPARISON_GROUP, Treatment(reference='Descanso igual'))
   + C(HOME_AWAY_LABEL, Treatment(reference='Fora'))
   + TEAM_WIN_PCT_BEFORE
   + OPP_WIN_PCT_BEFORE
   + TEAM_AVG_PLUS_MINUS_BEFORE
   + OPP_AVG_PLUS_MINUS_BEFORE
"""

modelo_ols_saldo = fit_ols_clustered(formula_ols_saldo, model_df)
coef_table(modelo_ols_saldo)


#### Comparação geral dos modelos

A tabela abaixo ajuda a comparar os modelos logísticos. Em geral, menor AIC sugere melhor ajuste relativo, mas a decisão final deve considerar também interpretabilidade e coerência metodológica.

In [ ]:
comparacao_modelos_logit = pd.DataFrame({
    'modelo': [
        '1. Descanso apenas',
        '2. Descanso + casa/fora',
        '3. Ajustado por força prévia',
        '4. Interação descanso × casa/fora'
    ],
    'n_obs': [
        int(modelo_logit_simples.nobs),
        int(modelo_logit_casa.nobs),
        int(modelo_logit_ajustado.nobs),
        int(modelo_logit_interacao.nobs)
    ],
    'AIC': [
        modelo_logit_simples.aic,
        modelo_logit_casa.aic,
        modelo_logit_ajustado.aic,
        modelo_logit_interacao.aic
    ],
    'pseudo_R2_McFadden': [
        get_pseudo_r2(modelo_logit_simples),
        get_pseudo_r2(modelo_logit_casa),
        get_pseudo_r2(modelo_logit_ajustado),
        get_pseudo_r2(modelo_logit_interacao)
    ]
})

comparacao_modelos_logit


#### Análise complementar com `REST_MATCHUP`

Esta análise usa a combinação detalhada do descanso do time contra o descanso do adversário. Ela é útil como complemento exploratório, mas não deve substituir o modelo principal, porque a categoria `2+` compacta várias situações diferentes.

In [ ]:
# Modelo complementar com REST_MATCHUP detalhado
# Esta análise é exploratória e mais granular que o modelo principal.
# Como model_df mantém a perspectiva do time com menor descanso ou descanso igual,
# algumas categorias de REST_MATCHUP não aparecem. Se elas ficarem como categorias
# vazias, o statsmodels pode gerar erro de matriz singular.
# Por isso, criamos uma variável limpa apenas com categorias realmente observadas.

matchup_model_df = model_df.copy()

# Transformando em string para remover categorias não observadas herdadas do Categorical original
matchup_model_df['REST_MATCHUP_MODEL'] = matchup_model_df['REST_MATCHUP'].astype(str)

# Mantendo apenas a ordem das categorias que realmente aparecem na base filtrada
observed_matchup_order = [
    cat for cat in rest_matchup_order
    if cat in matchup_model_df['REST_MATCHUP_MODEL'].unique()
]

matchup_model_df = matchup_model_df[
    matchup_model_df['REST_MATCHUP_MODEL'].isin(observed_matchup_order)
].copy()

# Tabela de checagem: quantidade de observações e taxa de vitória por categoria
matchup_check = (
    matchup_model_df
    .groupby('REST_MATCHUP_MODEL')
    .agg(
        jogos=('GAME_ID', 'count'),
        vitorias=('WL', 'sum'),
        taxa_vitoria=('WL', 'mean')
    )
    .reindex(observed_matchup_order)
)

matchup_check['taxa_vitoria'] = matchup_check['taxa_vitoria'] * 100
print('Categorias observadas no modelo complementar REST_MATCHUP:')
display(matchup_check)

# Mantemos apenas categorias com pelo menos duas classes de desfecho.
# Se uma categoria tiver só vitórias ou só derrotas, o Logit pode sofrer separação perfeita.
valid_matchups = matchup_check[
    (matchup_check['jogos'] > 0) &
    (matchup_check['vitorias'] > 0) &
    (matchup_check['vitorias'] < matchup_check['jogos'])
].index.tolist()

# Garante uma categoria de referência válida.
# Preferimos 0 vs 0, quando disponível, por ser a referência mais intuitiva.
if '0 vs 0' in valid_matchups:
    matchup_reference = '0 vs 0'
elif len(valid_matchups) > 0:
    matchup_reference = valid_matchups[0]
else:
    raise ValueError('Nenhuma categoria REST_MATCHUP válida para ajuste do modelo complementar.')

matchup_model_df = matchup_model_df[
    matchup_model_df['REST_MATCHUP_MODEL'].isin(valid_matchups)
].copy()

matchup_model_df['REST_MATCHUP_MODEL'] = pd.Categorical(
    matchup_model_df['REST_MATCHUP_MODEL'],
    categories=valid_matchups,
    ordered=True
)
matchup_model_df['REST_MATCHUP_MODEL'] = matchup_model_df['REST_MATCHUP_MODEL'].cat.remove_unused_categories()

print(f'Categoria de referência usada no modelo complementar: {matchup_reference}')

formula_logit_matchup = f'''
WL ~ C(REST_MATCHUP_MODEL, Treatment(reference="{matchup_reference}"))
   + C(HOME_AWAY_LABEL, Treatment(reference='Fora'))
   + TEAM_WIN_PCT_BEFORE
   + OPP_WIN_PCT_BEFORE
   + TEAM_AVG_PLUS_MINUS_BEFORE
   + OPP_AVG_PLUS_MINUS_BEFORE
'''

modelo_logit_matchup = fit_logit_clustered(formula_logit_matchup, matchup_model_df)
odds_ratio_table(modelo_logit_matchup)


### Interpretação sugerida

Para o trabalho, o modelo principal mais defensável é o **Modelo 3**, porque avalia a diferença real de descanso, ajusta para mando de quadra e controla a força prévia do time e do oponente. O **Modelo 4** deve ser usado para verificar se o efeito do descanso é diferente em jogos em casa versus fora. O modelo com `REST_MATCHUP` fica como análise complementar/exploratória.